# Обучение: пример запуска

Надо скопировать этот ноутбук и выполнить общие ячейки, затем выбрать **одну из трёх веток**: с нуля, с весов или resume. У каждой ветки свои параметры, сетап и запуск. Для последовательного запуска нескольких новых экспериментов предусмотрен отдельный блок очереди в конце; он выполняется вместо одиночной ветки.

Обучение запускается на двух GPU T4 в Kaggle. Метрики отправляются в W&B, результаты сохраняются в `/kaggle/working/experiment_runs`.

## Общие пути

- `PROJECT_PATH`: репозиторий с `utilities/train_script.py` и конфигами.
- `DATA_PATH`: исходная папка `tokenized` с `dataset_dict.json`, `tokenizer.json`, `train` и `val`. Используется при создании нового эксперимента и подготовке отсутствующих данных для resume.
- `WORK_DIR`: рабочая папка Kaggle для конфигов, данных и результатов.

Имя эксперимента и имя конфига задаются отдельно в ветках 1 и 2. Начальные веса нужны только ветке 2, папка сохранённого эксперимента — только ветке 3.

In [ ]:
from pathlib import Path

PROJECT_PATH = Path("/kaggle/input/datasets/wwip62/nlp-hw2-codegen").expanduser().resolve()
DATA_PATH = Path("/kaggle/input/datasets/wwip62/codegen-tokenized-bpe-16384/tokenized").expanduser().resolve()
WORK_DIR = Path("/kaggle/working").resolve()

TOKENIZER_PATH = DATA_PATH / "tokenizer.json"
DATA_DIR = WORK_DIR / "tokenized"

In [ ]:
if not (PROJECT_PATH / "utilities" / "train_script.py").is_file():
    raise FileNotFoundError(f"Скрипт обучения не найден в репозитории: {PROJECT_PATH}")

print("Репозиторий:", PROJECT_PATH)
print("Исходные данные:", DATA_PATH)
print("Рабочая папка:", WORK_DIR)

In [ ]:
%pip install -q datasets tokenizers tqdm PyYAML "sacrebleu>=2.4,<3" wandb

## Импорты

In [ ]:
import os
import shutil
import subprocess
import sys

import torch
import yaml
from kaggle_secrets import UserSecretsClient

## Общие настройки

В Kaggle надо включить **GPU T4 ×2**. `NUM_GPUS` и W&B используются во всех ветках. Параметры LR и warmup применяются только к новым экспериментам в ветках 1 и 2. Resume восстанавливает настройки из сохранённого эксперимента.

In [ ]:
NUM_GPUS = 2

# Если надо, можно переопределить
LR = None
MIN_LR = None
WARMUP_STEPS = None
WARMUP_START_FACTOR = None

WANDB_PROJECT = "nlp-hw2"
WANDB_ENTITY = "62wip-hse-ml"

## Настройка W&B

In [ ]:
env = os.environ.copy()
env["PYTHONUNBUFFERED"] = "1"
env["WANDB_MODE"] = "online"
# Надо проставить WANDB_API_KEY в Kaggle Secrets
env["WANDB_API_KEY"] = UserSecretsClient().get_secret("WANDB_API_KEY")

## Подготовка данных и нового эксперимента

`prepare_data` копирует train и валидацию в рабочую папку `/kaggle/working/tokenized`: обучение читает локальную копию, а кеш валидации записывается рядом с данными. Копирование train требует около 8 ГиБ диска и времени перед запуском. Функция вызывается при новом эксперименте и при отсутствии данных или старой ссылке на train для resume.

`prepare_train_config` используется только в ветках 1 и 2: создаёт конфиг с абсолютными путями и именем эксперимента. Параметры warmup переопределяются независимо; `None` сохраняет соответствующее значение исходного конфига.

In [ ]:
def prepare_data(data_dir=DATA_DIR):
    dataset_file = DATA_PATH / "dataset_dict.json"
    if not dataset_file.is_file():
        raise FileNotFoundError(f"Файл данных не найден: {dataset_file}")
    for split in ("train", "val"):
        if not (DATA_PATH / split).is_dir():
            raise FileNotFoundError(f"Выборка не найдена: {DATA_PATH / split}")

    data_dir.mkdir(parents=True, exist_ok=True)
    shutil.copy2(DATA_PATH / "dataset_dict.json", data_dir / "dataset_dict.json")
    shutil.copytree(DATA_PATH / "val", data_dir / "val", dirs_exist_ok=True)

    train_path = data_dir / "train"
    if train_path.is_symlink():
        train_path.unlink()
    shutil.copytree(DATA_PATH / "train", train_path, dirs_exist_ok=True)


def prepare_train_config(run_name, config_name):
    config_source = PROJECT_PATH / "experiment_configs" / config_name
    config_path = WORK_DIR / f"{run_name}_config.yaml"
    run_path = WORK_DIR / "experiment_runs" / run_name
    if run_path.exists():
        raise FileExistsError(f"Эксперимент уже существует: {run_path}. Нужно другое RUN_NAME или режим resume.")
    config = yaml.safe_load(config_source.read_text())
    if not TOKENIZER_PATH.is_file():
        raise FileNotFoundError(f"Токенайзер не найден: {TOKENIZER_PATH}")
    prepare_data()

    config["name"] = run_name
    config["data"]["tokenized_dir"] = str(DATA_DIR)
    config["data"]["tokenizer_path"] = str(TOKENIZER_PATH)
    if LR is not None:
        config["optimizer"]["params"]["lr"] = LR
    if MIN_LR is not None:
        config["scheduler"]["params"]["eta_min"] = MIN_LR
    if WARMUP_STEPS is not None or WARMUP_START_FACTOR is not None:
        warmup = config["scheduler"].get("warmup") or {}
        if WARMUP_STEPS is not None:
            warmup["steps"] = WARMUP_STEPS
        if WARMUP_START_FACTOR is not None:
            warmup["start_factor"] = WARMUP_START_FACTOR
        config["scheduler"]["warmup"] = warmup
    config_path.write_text(yaml.safe_dump(config, sort_keys=False))

    print("Конфиг:", config_path)
    print("Данные:", DATA_DIR)
    print("Токенайзер:", TOKENIZER_PATH)
    print("Результаты:", run_path)
    return config_path, run_path

## 1. Setup

Новый эксперимент со случайными весами модели и новым оптимизатором. Надо указать имя эксперимента и имя YAML-файла из `experiment_configs`.

In [ ]:
RUN_NAME = ...  # TODO: имя нового эксперимента
CONFIG_NAME = ...  # TODO: имя файла конфига

In [ ]:
CONFIG_PATH, RUN_PATH = prepare_train_config(RUN_NAME, CONFIG_NAME)

In [ ]:
assert torch.cuda.device_count() >= NUM_GPUS, "Для запуска требуется GPU T4 ×2 в настройках Kaggle."

command = [
    sys.executable, "-m", "torch.distributed.run",
    "--standalone", f"--nproc_per_node={NUM_GPUS}",
    "-m", "utilities.train_script",
    "--config", str(CONFIG_PATH),
    "--wandb-project", WANDB_PROJECT,
]

if WANDB_ENTITY:
    command += ["--wandb-entity", WANDB_ENTITY]

subprocess.run(
    command,
    cwd=PROJECT_PATH,
    env={**env, "PROJECT_DIR": str(WORK_DIR)},
    check=True,
)

## 2. Setup + weights reuse

Новый эксперимент с весами из `WEIGHTS_FROM`. Оптимизатор и scheduler создаются заново. Надо указать имя нового эксперимента, конфиг и полный путь `PATH_TO_EXPERIMENT` к исходному эксперименту. `CHECKPOINT` выбирает начальные веса: `best.pt` или `last.pt`.

In [ ]:
RUN_NAME = ...  # TODO: имя нового эксперимента
CONFIG_NAME = ...  # TODO: имя файла конфига

PATH_TO_EXPERIMENT = ...  # TODO: полный путь к папке исходного эксперимента
CHECKPOINT = "best"  # "best" или "last"

if PATH_TO_EXPERIMENT is ...:
    raise ValueError("Надо задать PATH_TO_EXPERIMENT: полный путь к папке исходного эксперимента.")
if CHECKPOINT not in ("best", "last"):
    raise ValueError('CHECKPOINT должен быть "best" или "last".')

EXPERIMENT_PATH = Path(PATH_TO_EXPERIMENT).expanduser().resolve()
WEIGHTS_FROM = EXPERIMENT_PATH / "checkpoints" / f"{CHECKPOINT}.pt"

In [ ]:
if not WEIGHTS_FROM.is_file():
    raise FileNotFoundError(f"Начальные веса не найдены: {WEIGHTS_FROM}")

CONFIG_PATH, RUN_PATH = prepare_train_config(RUN_NAME, CONFIG_NAME)

In [ ]:
assert torch.cuda.device_count() >= NUM_GPUS, "Для запуска требуется GPU T4 ×2 в настройках Kaggle."

command = [
    sys.executable, "-m", "torch.distributed.run",
    "--standalone", f"--nproc_per_node={NUM_GPUS}",
    "-m", "utilities.train_script",
    "--config", str(CONFIG_PATH),
    "--weights-from", str(WEIGHTS_FROM),
    "--wandb-project", WANDB_PROJECT,
]

if WANDB_ENTITY:
    command += ["--wandb-entity", WANDB_ENTITY]

subprocess.run(
    command,
    cwd=PROJECT_PATH,
    env={**env, "PROJECT_DIR": str(WORK_DIR)},
    check=True,
)

## 3. Resume

Продолжение существующего эксперимента из `checkpoints/last.pt`: восстанавливаются модель, оптимизатор, scheduler и прогресс. Надо явно задать полный путь `PATH_TO_EXPERIMENT` к папке эксперимента, доступной для записи. Resume использует `last.pt`.

Новый конфиг, `RUN_NAME`, `CONFIG_NAME` и `WEIGHTS_FROM` не нужны. Токенайзер берётся из `init_state/tokenizer.json`, данные — по пути из сохранённого `config.yaml`. Если данных по сохранённому пути ещё нет, они подготавливаются из `DATA_PATH`. Исходный токенайзер для этого не требуется. Число GPU должно совпадать с сохранённым запуском. При наличии `init_state/wandb.json` продолжается тот же запуск W&B.

In [ ]:
PATH_TO_EXPERIMENT = ...  # TODO: полный путь к папке эксперимента для продолжения

if PATH_TO_EXPERIMENT is ...:
    raise ValueError("Надо задать PATH_TO_EXPERIMENT: полный путь к папке эксперимента для продолжения.")

RESUME_PATH = Path(PATH_TO_EXPERIMENT).expanduser().resolve()

In [ ]:
required_resume_files = [
    RESUME_PATH / "config.yaml",
    RESUME_PATH / "init_state" / "tokenizer.json",
    RESUME_PATH / "init_state" / "sample_indices.json",
    RESUME_PATH / "checkpoints" / "last.pt",
]
missing_files = [str(path) for path in required_resume_files if not path.is_file()]
if missing_files:
    raise FileNotFoundError("Некорректный RESUME_PATH. Не найдены файлы:\n" + "\n".join(missing_files))


resume_config = yaml.safe_load((RESUME_PATH / "config.yaml").read_text())
resume_data_dir = (WORK_DIR / Path(resume_config["data"]["tokenized_dir"])).resolve()
if (
    not (resume_data_dir / "dataset_dict.json").is_file()
    or (resume_data_dir / "train").is_symlink()
    or not all((resume_data_dir / split).is_dir() for split in ("train", "val"))
):
    prepare_data(resume_data_dir)

print("Продолжение:", RESUME_PATH)
print("Данные:", resume_data_dir)

In [ ]:
assert torch.cuda.device_count() >= NUM_GPUS, "Для запуска требуется GPU T4 ×2 в настройках Kaggle."

command = [
    sys.executable, "-m", "torch.distributed.run",
    "--standalone", f"--nproc_per_node={NUM_GPUS}",
    "-m", "utilities.train_script",
    "--resume", str(RESUME_PATH),
    "--wandb-project", WANDB_PROJECT,
]

if WANDB_ENTITY:
    command += ["--wandb-entity", WANDB_ENTITY]

subprocess.run(
    command,
    cwd=PROJECT_PATH,
    env={**env, "PROJECT_DIR": str(WORK_DIR)},
    check=True,
)

## 4. Очередь экспериментов с лимитом времени

После общих ячеек надо заранее указать все пути к YAML-конфигам в `CONFIG_PATHS` и лимит `MAX_TRAIN_SECONDS` на каждый эксперимент. Число экспериментов определяется длиной списка; порядок запуска совпадает с порядком путей. Пример ниже запускает три пилота с бюджетом 3600 секунд на каждый.

Каждый эксперимент создаётся с нуля. Имя и настройки обучения читаются из его YAML; общие переопределения LR и warmup из одиночных веток здесь не применяются. Пути к данным и токенайзеру заменяются на рабочие абсолютные пути Kaggle. Данные подготавливаются один раз, а все конфиги проверяются до начала обучения.

Используются `NUM_GPUS` и общие настройки W&B. Эксперименты запускаются последовательно; при первой ошибке очередь останавливается. Лимит передаётся через `--max-train-seconds`: финальная валидация и сохранение могут завершиться после истечения бюджета.

In [ ]:
CONFIG_PATHS = [
    PROJECT_PATH / "experiment_configs/1_hour_exps/pilot_1h_control.yaml",
    PROJECT_PATH / "experiment_configs/1_hour_exps/pilot_1h_enc7_dec5.yaml",
    PROJECT_PATH / "experiment_configs/1_hour_exps/pilot_1h_ffn1536.yaml",
]
MAX_TRAIN_SECONDS = 3600

In [ ]:
if not CONFIG_PATHS:
    raise ValueError("CONFIG_PATHS должен содержать хотя бы один путь к конфигу.")

experiments = []
run_names = set()
for source_path in CONFIG_PATHS:
    source_path = Path(source_path).expanduser().resolve()
    config = yaml.safe_load(source_path.read_text())
    run_name = config["name"]
    if not isinstance(run_name, str) or not run_name:
        raise ValueError(f"В конфиге не задано имя эксперимента: {source_path}")
    if run_name in run_names:
        raise ValueError(f"Имя эксперимента повторяется в очереди: {run_name}")
    run_names.add(run_name)

    run_path = WORK_DIR / "experiment_runs" / run_name
    if run_path.exists():
        raise FileExistsError(f"Эксперимент уже существует: {run_path}")
    config["data"]["tokenized_dir"] = str(DATA_DIR)
    config["data"]["tokenizer_path"] = str(TOKENIZER_PATH)
    config_path = WORK_DIR / f"{run_name}_config.yaml"
    experiments.append((config_path, config, run_path))

if not TOKENIZER_PATH.is_file():
    raise FileNotFoundError(f"Токенайзер не найден: {TOKENIZER_PATH}")
prepare_data()

for config_path, config, run_path in experiments:
    config_path.write_text(yaml.safe_dump(config, sort_keys=False))
    print(f"{config['name']}: {config_path} → {run_path}")

print(f"В очереди: {len(experiments)}; бюджет на эксперимент: {MAX_TRAIN_SECONDS} с")

In [ ]:
assert torch.cuda.device_count() >= NUM_GPUS, "Для запуска требуется GPU T4 ×2 в настройках Kaggle."

for number, (config_path, config, run_path) in enumerate(experiments, start=1):
    print(f"[{number}/{len(experiments)}] Запуск: {config['name']}", flush=True)
    command = [
        sys.executable, "-u", "-m", "torch.distributed.run",
        "--standalone", f"--nproc_per_node={NUM_GPUS}",
        "-m", "utilities.train_script",
        "--device", "cuda",
        "--config", str(config_path),
        "--max-train-seconds", str(MAX_TRAIN_SECONDS),
        "--wandb-project", WANDB_PROJECT,
    ]
    if WANDB_ENTITY:
        command += ["--wandb-entity", WANDB_ENTITY]

    result = subprocess.run(
        command,
        cwd=PROJECT_PATH,
        env={**env, "PROJECT_DIR": str(WORK_DIR)},
        check=False,
    )
    if result.returncode != 0:
        print(f"Очередь остановлена: {config['name']}, код выхода {result.returncode}.", flush=True)
        break
    print(f"Завершён: {config['name']} -> {run_path}", flush=True)
else:
    print("Все эксперименты в очереди завершены.")

## 5. Удаление рабочей копии данных

Эта ячейка удаляет `/kaggle/working/tokenized` вместе с кешем данных. Чекпойнты и метрики в `experiment_runs` сохраняются. При следующем запуске или resume данные будут скопированы заново.

In [ ]:
cleanup_dir = WORK_DIR / "tokenized"
if cleanup_dir.is_symlink():
    cleanup_dir.unlink()
elif cleanup_dir.exists():
    shutil.rmtree(cleanup_dir)
print("Рабочая копия данных удалена:", cleanup_dir)